# Persistent Community ID Assignment

This notebook runs and inspects the first step of stage identification for the cumulative, interval, and overlapping snapshot approaches.

Identity inheritance uses:

- Jaccard threshold: `0.5`
- Prospective stability: overlap / previous community size
- Retrospective stability: overlap / current community size
- Stability threshold: `0.5`
- Unique mutual nomination for one-to-one ID inheritance
- New IDs for exact ties or relationships below either threshold

In [1]:
from pathlib import Path
import sys

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "graph-matching":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "graph-matching"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from identity_events import run_approach

INPUT_DIR = NOTEBOOK_DIR / "outputs" / "graph_matching"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "stage_identification"
APPROACHES = ["cumulative", "interval", "overlap"]
JACCARD_THRESHOLD = 0.5
STABILITY_THRESHOLD = 0.5

## Generate Identified Community Histories

In [2]:
for approach in APPROACHES:
    run_approach(
        input_dir=INPUT_DIR,
        output_dir=OUTPUT_DIR,
        approach=approach,
        jaccard_threshold=JACCARD_THRESHOLD,
        stability_threshold=STABILITY_THRESHOLD,
    )

print(f"Outputs written to {OUTPUT_DIR}")

Outputs written to /Users/maha_personal/Uni/SS 26/IDP/code/graph-matching/outputs/stage_identification


## Summarize Identity Continuity

In [3]:
summary_rows = []
for approach in APPROACHES:
    communities = pd.read_csv(OUTPUT_DIR / approach / "identified_communities.csv")
    transitions = pd.read_csv(OUTPUT_DIR / approach / "identity_transitions.csv")
    lifespan_lengths = communities.groupby("persistent_id").size()

    summary_rows.append(
        {
            "approach": approach,
            "community_observations": len(communities),
            "persistent_ids": communities["persistent_id"].nunique(),
            "inherited_transitions": (transitions["decision"] == "inherited").sum(),
            "new_id_transitions": (transitions["decision"] == "new").sum(),
            "multi_snapshot_ids": (lifespan_lengths > 1).sum(),
            "longest_lifespan_snapshots": lifespan_lengths.max(),
        }
    )

summary = pd.DataFrame(summary_rows)
summary

,approach,community_observations,persistent_ids,inherited_transitions,new_id_transitions,multi_snapshot_ids,longest_lifespan_snapshots
0,cumulative,147,35,112,20,19,11
1,interval,159,94,65,79,28,11
2,overlap,289,119,170,104,57,20


## Inspect Transition Decisions

The transition table preserves the winning stability values and candidate parent IDs so inheritance decisions remain auditable.

In [4]:
approach_to_inspect = "interval"
transitions = pd.read_csv(
    OUTPUT_DIR / approach_to_inspect / "identity_transitions.csv"
)
transitions.head(15)

,approach,snapshot_index,local_id,persistent_id,decision,from_snapshot,from_local_id,from_persistent_id,overlap_size,jaccard,prospective_stability,retrospective_stability,candidate_parent_local_ids
0,interval,1,0,INT-C0001,inherited,0.0,0.0,INT-C0001,65.0,0.515873,0.590909,0.802469,0.0
1,interval,1,1,INT-C0016,new,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,interval,1,2,INT-C0002,inherited,0.0,1.0,INT-C0002,64.0,0.609524,0.640000,0.927536,1.0
3,interval,1,3,INT-C0003,inherited,0.0,2.0,INT-C0003,44.0,0.511628,0.721311,0.637681,2.0
4,interval,1,4,INT-C0017,new,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,interval,1,5,INT-C0006,inherited,0.0,5.0,INT-C0006,46.0,0.613333,0.807018,0.718750,5.0
6,interval,1,6,INT-C0018,new,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,interval,1,7,INT-C0019,new,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,interval,1,8,INT-C0009,inherited,0.0,8.0,INT-C0009,36.0,0.692308,0.765957,0.878049,8.0
9,interval,1,9,INT-C0012,inherited,0.0,11.0,INT-C0012,25.0,0.625000,0.925926,0.657895,11.0
